In [2]:
from pathlib import Path
import json, requests
from utils import Helper
import pandas as pd
from datetime import datetime
helper = Helper()

In [ ]:
nse = r"https://www.nseindia.com/api/corporate-share-holdings-master?index=equities&from_date=13-10-2026&to_date=14-10-2026"
bse = r"https://api.bseindia.com/BseIndiaAPI/api/Corp_Shareholding_ng/w?scripcode=&flag=1&indtype="



In [ ]:
#SAVE JSON NSE
out_dir = Path("shareholding_json")
out_dir.mkdir(exist_ok=True)

# Common headers
headers_nse = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Referer": "https://www.nseindia.com/",
    "Connection": "keep-alive",
}

# NSE session
nse_session = requests.Session()
nse_session.get("https://www.nseindia.com", headers=headers_nse, timeout=15, verify=False)

nse_url = "https://www.nseindia.com/api/corporate-share-holdings-master"
payload_nse = {
    "index":"equities",
    "from_date":"13-10-2026",
    "to_date":"14-10-2026"
}

nse_resp = nse_session.get(nse_url, params=payload_nse, headers=headers_nse, timeout=30, verify=False)
nse_resp.raise_for_status()
nse_data = nse_resp.json()

with open(out_dir / "nse_shareholding.json", "w", encoding="utf-8") as f:
    json.dump(nse_data, f, indent=2)
print("NSE JSON saved")



In [ ]:
import requests
from pathlib import Path
import utils  # assuming your utils.load_json is available

# Output directory
out_dir = Path("shareholding_nse")
out_dir.mkdir(exist_ok=True)

# Headers for NSE
headers_nse = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0.0.0 Safari/537.36"
    ),
    "Accept": "application/xml, text/html, */*; q=0.01",
    "Referer": "https://www.nseindia.com/",
    "Connection": "keep-alive",
}

# Establish session
nse_session = requests.Session()
nse_session.get("https://www.nseindia.com", headers=headers_nse, timeout=15, verify=False)

# Load JSON file
json_dir = r"C:\Users\kaustubh.keny\Projects\RANDOM_FETCH_DATA\bse_share\nse_sample.json"
data = helper.load_json(json_dir)

# Iterate and download XMLs
for content in data:
    isin = content["isin"]
    xbrl_link = content["xbrl"]
    date_str = content["date"].replace("-", "")  # e.g. 30SEP2026 → 30SEP2026 or normalize
    
    # Build filename: date_isin.xml
    filename = f"{date_str}_{isin}.xml"
    out_file = out_dir / filename

    try:
        resp = nse_session.get(xbrl_link, headers=headers_nse, timeout=30, verify=False)
        resp.raise_for_status()
        with open(out_file, "wb") as f:
            f.write(resp.content)
        print(f"✅ Saved {out_file}")
    except Exception as e:
        print(f"⚠️ Failed for {isin}: {e}")


In [ ]:
#SAVE JSON BSE
import requests
import json
from pathlib import Path

out_dir = Path("shareholding_json")
out_dir.mkdir(exist_ok=True)

# Define headers first
headers_bse = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Referer": "https://www.bseindia.com/",
    "Connection": "keep-alive",
    "Accept-Encoding": "gzip, deflate, br",
    "Host": "api.bseindia.com"
}

# Start session and hit homepage to establish cookies
bse_session = requests.Session()
bse_session.get("https://www.bseindia.com/", headers=headers_bse, timeout=15, verify=False)

# API URL + payload
bse_url = "https://api.bseindia.com/BseIndiaAPI/api/Corp_Shareholding_ng/w"
payload_bse = {
    "scripcode": "",
    "flag": "1",
    "indtype": ""
}

# Call API with cookies + headers
bse_resp = bse_session.get(bse_url, params=payload_bse, headers=headers_bse, timeout=30, verify=False)
bse_resp.raise_for_status()
bse_data = bse_resp.json()

with open(out_dir / "bse_shareholding.json", "w", encoding="utf-8") as f:
    json.dump(bse_data, f, indent=2)

print("BSE JSON saved")


In [ ]:
import requests, time
from pathlib import Path

# Output directory
out_dir_bse = Path("shareholding_bse")
out_dir_bse.mkdir(exist_ok=True)

# Headers for BSE
headers_bse = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://www.bseindia.com/",
    "Connection": "keep-alive",
}

# Establish session
bse_session = requests.Session()
bse_session.get("https://www.bseindia.com", headers=headers_bse, timeout=15, verify=False)

# Load JSON file
json_dir_bse = r"C:\Users\kaustubh.keny\Projects\RANDOM_FETCH_DATA\bse_share\bse_sample.json"
data_bse = helper.load_json(json_dir_bse)

# Iterate and download HTML attachments
for content in data_bse["Table"]:
    scripcode = content["FLD_ScripCode"]
    end_date = content["EndDate"].split("T")[0].replace("-", "")  # e.g. 20260930
    attachment = content["XBRLAttachment"]

    # Build full URL
    xbrl_link = f"https://www.bseindia.com{attachment}"

    # Filename: date_scripcode.html
    filename = f"{end_date}_{scripcode}.html"
    out_file = out_dir_bse / filename

    try:
        resp = bse_session.get(xbrl_link, headers=headers_bse, timeout=30, verify=False)
        resp.raise_for_status()
        with open(out_file, "wb") as f:
            f.write(resp.content)
        print(f"Saved {out_file}")
        time.sleep(.2)
    except Exception as e:
        print(f"Failed for {scripcode}: {e}")


In [ ]:
from bs4 import BeautifulSoup
from pathlib import Path
from openpyxl import Workbook
from openpyxl.styles import Alignment

html_dir = Path(r"C:\Users\kaustubh.keny\Projects\RANDOM_FETCH_DATA\bse_share\shareholding_bse")
out_dir = Path("parsed_excel")
out_dir.mkdir(parents=True, exist_ok=True)

sheets = {}

def get_span(cell, attr):
    try:
        return max(1, int(cell.get(attr, 1) or 1))
    except (ValueError, TypeError):
        return 1

for html_file in html_dir.glob("*.html"):
    with open(html_file, encoding="utf-8", errors="ignore") as f:
        soup = BeautifulSoup(f, "html.parser")

    tables = soup.find_all("table")
    print(f"{html_file.name}: {len(tables)} tables")

    if not tables:
        continue

    wb = Workbook()
    wb.remove(wb.active)

    import re

    def clean_sheet_name(name):
        # Excel sheet names cannot contain: \ / * ? : [ ]
        name = re.sub(r'[\\/*?:\[\]]', '_', name)
        return name[:31]  # Excel limit

    for idx, table in enumerate(tables, 1):

        # Find nearest preceding heading
        title = None

        prev = table.find_previous(["h1", "h2", "h3"])
        if prev:
            title = prev.get_text(" ", strip=True)

        if not title:
            title = f"Table_{idx}"

        ws = wb.create_sheet(clean_sheet_name(title))

        # Optional: write heading above table
        ws.cell(1, 1, title)
        ws.cell(1, 1).alignment = Alignment(wrap_text=True)

        occupied = set()

        rows = [
            tr for tr in table.find_all("tr")
            if tr.find_parent("table") is table
        ]

        excel_row_offset = 1  # table starts from row 2

        for r, tr in enumerate(rows, 1):
            r += excel_row_offset
            c = 1

            for cell in tr.find_all(["td", "th"], recursive=False):

                while (r, c) in occupied:
                    c += 1

                rowspan = get_span(cell, "rowspan")
                colspan = get_span(cell, "colspan")

                value = cell.get_text(separator="", strip=False)

                ws.cell(r, c, value)

                for rr in range(r, r + rowspan):
                    for cc in range(c, c + colspan):
                        occupied.add((rr, cc))

                if rowspan > 1 or colspan > 1:
                    ws.merge_cells(
                        start_row=r,
                        start_column=c,
                        end_row=r + rowspan - 1,
                        end_column=c + colspan - 1
                    )

                ws.cell(r, c).alignment = Alignment(
                    vertical="top",
                    wrap_text=True
                )

                c += colspan
    out_file = out_dir / f"{html_file.stem}.xlsx"
    wb.save(out_file)
    print(f"Saved: {out_file} ({len(tables)} sheets)")
    sheets.update({html_file.stem: len(tables)})

helper.save_json(sheets, "table_count.json")